In [72]:
from openai import OpenAI
from dotenv import load_dotenv

import os
import json
from IPython.display import Markdown

load_dotenv(override=True)

True

In [73]:
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)

In [74]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [75]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [76]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [ ]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    while True:
        completion = llm_client.chat.completions.create(
                model=MODEL, 
                messages=messages, 
                tools=tools
            )
           
        reply = completion.choices[0].message
        messages.append(reply)
    
        # reply_json = reply.model_dump(exclude_none=True)
        # reply_json.pop("reasoning", None)            # hide the model's long thinking
        # reply_json.pop("reasoning_details", None)
        # print("LLM REPLY:", json.dumps(reply_json, indent=2))
    
        if not reply.tool_calls: 
            return Markdown(reply.content)
            break                        # model asked for a tool
       
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})        

In [78]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

Tool called: validate_order_id {'order_id': 'OR-1045'} True
Tool called: log_complaint {'order_id': 'OR-1045', 'phone': '0322342423', 'complaint': 'received roll instead of burger', 'customer_name': 'Sam'} {'message': 'Saved! Total Complaint: 1'}


Hey Sam, I'm really sorry about the mix‑up with your order – receiving a roll instead of a burger is definitely not what we aim for. 🌟 I've logged your complaint (order OR‑1045) and our team will look into it right away.

Is there anything else I can help you with today, or would you like us to replace the item or issue a refund?

In [79]:
print(json.dumps(complaint_list))

[{"customer_name": "Sam", "order_id": "OR-1045", "phone": "0322342423", "complaint": "received roll instead of burger", "status": "Pending"}]




Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID